# 05 — Train the sequence classifier
Missing sample caches are built first. Training cells start a new run. Sample intact narration-containing sequences more often. Only projections, fusion, learned embeddings, Transformer and classifier train.


**Padded-box experiment:** retains original DETR features and adds frozen local DINOv2 crop features. Uses this folder's caches, split snapshot, studies and checkpoints.


In [1]:
# Reuse missing base caches in this isolated experiment before checking prerequisites.
from config import reuse_base_caches
reuse_base_caches()

# Cell purpose: Load validated training samples and prepare the original experiment output paths.
# Inputs: Existing feature/sample caches and training settings.
# Outputs: Training data and checkpoint location; the training cells below start a fresh run.

from pathlib import Path
import json
from config import CONFIG, ROOT, fingerprint, save_json, require_cache, run_stage, model_files

# Edit config.py once. Restart the kernel after changing it.
# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
CACHE.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print('Existing caches:', [p.name for p in CACHE.iterdir() if p.is_file()])

import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from model import BoxClassifier, collate, metrics, prediction_rows, grad_norm
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'samples.pt', '04_build_training_samples.ipynb')
run_stage('04_build_training_samples.ipynb')
data = torch.load(CACHE / 'samples.pt', map_location='cpu', weights_only=True)
CHECKPOINT = RUN / 'best.pt'
print('Existing checkpoint:', CHECKPOINT.exists(), '| Training below starts a fresh run')

Existing caches: ['base_cache_provenance.json', 'base_samples.pt', 'deberta.pt', 'dino.pt', 'discovery.json', 'magi.pt', 'samples.pt', 'sequences.json']
Running prerequisite: 04_build_training_samples.ipynb
Running prerequisite: 03a_cache_dino_crops.ipynb
Local encoder: C:\MODELS\dinov2-small
Padding per side: 0.2 | matching feature cache: True
DINO features: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\dino.pt | sequences: 80
Reuse matching padded samples: True
Exact original split: 64 train / 16 validation
Original train class counts: [1142, 50]
Feature widths: text 768, DETR 256, DINO 384, geometry 4, tail 2; fused input 688.
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\samples.pt
Existing checkpoint: False | Training below starts a fresh run


Set the seed, build whole-sequence loaders/model/loss, and define one training or validation epoch.


In [2]:
# Cell purpose: Set the seed, build whole-sequence loaders/model/loss, and define one training or validation epoch.
# Inputs: Saved split, cached features and original training-only class weights.
# Outputs: A weighted sequence sampler, small trainable classifier and run_epoch helper.

# Seed initialization and stochastic training; the already saved split is loaded separately.
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train = [data['samples'][i] for i in data['train_indices']]
val = [data['samples'][i] for i in data['val_indices']]
# Sequences with no labels remain in the cache and validation; they provide no training signal alone.
train = [s for s in train if (s['labels'] != -100).any()]
# A narration-containing full sequence gets higher sampling weight; individual boxes are never sampled alone.
sampling_weights = [CONFIG['narration_sampling_weight'] if (s['labels'] == 1).any() else 1.0 for s in train]
assert CONFIG['narration_sampling_weight'] > 0
# Replacement allows a complete sequence to appear multiple times in one epoch.
sampler = WeightedRandomSampler(sampling_weights, len(train), replacement=True,
                                generator=torch.Generator().manual_seed(CONFIG['seed']))
train_loader = DataLoader(train, batch_size=CONFIG['batch_size'], sampler=sampler, collate_fn=collate)
val_loader = DataLoader(val, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate)
visual_dim = train[0]['visual'].shape[-1]
model = BoxClassifier(visual_dim, data['max_positions'], crop_dim=train[0]['dino'].shape[-1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
weights = data['class_weights'].to(device)
# Sum weighted box losses, ignore -100 labels, then divide by labeled-box count below.
criterion = nn.CrossEntropyLoss(weight=weights, ignore_index=-100, reduction='sum')

# The same loop handles training and validation; optimizer steps occur only in training mode.
def run_epoch(loader, training):
    model.train(training)
    loss_sum, labeled_count, rows = 0., 0, []
    norms = []
    for batch in loader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        # Only labeled boxes enter the loss denominator; unlabeled real boxes still participate in attention.
        count = int((batch['labels'] != -100).sum())
        with torch.set_grad_enabled(training):
            logits = model(batch)
            summed = criterion(logits.reshape(-1, 2), batch['labels'].reshape(-1))
            if training and count:
                optimizer.zero_grad(set_to_none=True)
                # Normalize by labeled-box count, not by the sum of class weights.
                (summed / count).backward()
                # Log gradient magnitudes before clipping to make optimization diagnostics comparable.
                norms.append({'crop_projection': grad_norm(model.crop_projection),
                              'fusion': grad_norm(model.fusion), 'transformer': grad_norm(model.transformer),
                              'classifier': grad_norm(model.classifier)})
                # Clip the global trainable-parameter gradient norm before the optimizer update.
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        loss_sum += float(summed.detach())
        labeled_count += count
        # Export real boxes only; prediction_rows excludes padding but retains ignored-label context boxes.
        rows.extend(prediction_rows(batch, logits.detach().softmax(-1).cpu()))
    result = metrics(rows)
    result['loss'] = loss_sum / labeled_count if labeled_count else None
    if norms:
        result['grad_norm_mean_before_clip'] = {key: float(np.mean([n[key] for n in norms])) for key in norms[0]}
    return result, rows

Train for the configured epochs and retain the best validation macro-F1 checkpoint.


In [3]:
# Cell purpose: Train for the configured epochs and retain the best validation macro-F1 checkpoint.
# Inputs: The model, optimizer and loaders defined in the previous cell.
# Outputs: Epoch history, per-epoch predictions, runs/best.pt and best validation predictions.

history, best_f1 = [], -1.
for epoch in range(1, CONFIG['epochs'] + 1):
    train_metrics, _ = run_epoch(train_loader, True)
    val_metrics, rows = run_epoch(val_loader, False)
    record = {'epoch': epoch, 'train': train_metrics, 'val': val_metrics}
    history.append(record)
    print(json.dumps(record, indent=2))
    save_json(RUN / 'history.json', history)
    save_json(RUN / f'val_predictions_epoch_{epoch:02d}.json', rows)
    # Select by validation macro F1; a strict improvement keeps the earlier epoch when scores tie.
    if val_metrics['macro_f1'] > best_f1:
        best_f1 = val_metrics['macro_f1']
        torch.save({'state_dict': model.state_dict(), 'visual_dim': visual_dim,
            'max_positions': data['max_positions'], 'class_weights': data['class_weights'],
            'crop_dim': train[0]['dino'].shape[-1], 'feature_config': data['feature_config'],
            'samples_signature': data['signature'], 'epoch': epoch, 'metrics': val_metrics,
            'config': json.loads(json.dumps(CONFIG, default=str))}, CHECKPOINT)
        save_json(RUN / 'best_validation_predictions.json', rows)
print('Best validation macro F1:', best_f1, '| checkpoint:', CHECKPOINT)

{
  "epoch": 1,
  "train": {
    "macro_f1": 0.38607006276067,
    "balanced_accuracy": 0.5541666666666667,
    "SPOKEN": {
      "precision": 0.9,
      "recall": 0.35833333333333334,
      "f1": 0.5125827814569537,
      "support": 1080,
      "p_true_mean": 0.4306122234712044,
      "p_true_median": 0.40724650025367737,
      "p_true_gt_0.9": 0.008333333333333333,
      "p_true_lt_0.5": 0.6416666666666667
    },
    "NARRATED": {
      "precision": 0.15693430656934307,
      "recall": 0.75,
      "f1": 0.2595573440643863,
      "support": 172,
      "p_true_mean": 0.6393323772002099,
      "p_true_median": 0.6339975297451019,
      "p_true_gt_0.9": 0.06976744186046512,
      "p_true_lt_0.5": 0.25
    },
    "loss": 0.2411344706440886,
    "grad_norm_mean_before_clip": {
      "crop_projection": 0.21395036697604594,
      "fusion": 0.415055167186003,
      "transformer": 1.1091387866960947,
      "classifier": 1.4198035120651986
    }
  },
  "val": {
    "macro_f1": 0.483535528596187

Optional scratch cell; no operation is currently defined.


In [4]:
# Cell purpose: Optional scratch cell; no operation is currently defined.
# Inputs: None.
# Outputs: None; this cell contains comments only.

